In [1]:
import pandas as pd

orders = pd.read_csv("data/olist_orders_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
customers = pd.read_csv("data/olist_customers_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
category_translation = pd.read_csv("data/product_category_name_translation.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

print("all loaded")

all loaded


In [10]:
# section 1 - revenue overview

delivered = orders[orders["order_status"] == "delivered"].copy()
revenue = delivered.merge(order_items, on="order_id")

total_revenue = revenue["price"].sum().round(2)
total_freight = revenue["freight_value"].sum().round(2)
total_combined = (revenue["price"] + revenue["freight_value"]).sum().round(2)
total_orders = delivered["order_id"].nunique()

print("total revenue (delivered):", total_revenue)
print("total freight:", total_freight)
print("total combined:", total_combined)
print("total delivered orders:", total_orders)
canceled = len(orders[orders["order_status"] == "canceled"])
unavailable = len(orders[orders["order_status"] == "unavailable"])

lost_revenue = order_items[
    order_items["order_id"].isin(
        orders[orders["order_status"].isin(["canceled", "unavailable"])]["order_id"]
    )
]["price"].sum().round(2)

print("canceled orders:", canceled)
print("unavailable orders:", unavailable)
print("estimated lost revenue:", lost_revenue)

total revenue (delivered): 13221498.11
total freight: 2198275.64
total combined: 15419773.75
total delivered orders: 96478
canceled orders: 625
unavailable orders: 609
estimated lost revenue: 97242.96


In [5]:
# section 2 - monthly revenue trend

monthly = revenue.copy()
monthly["year"] = monthly["order_purchase_timestamp"].dt.year
monthly["month"] = monthly["order_purchase_timestamp"].dt.month

monthly_revenue = monthly.groupby(["year", "month"]).agg(
    revenue=("price", "sum"),
    total_orders=("order_id", "nunique")
).reset_index()

monthly_revenue["revenue"] = monthly_revenue["revenue"].round(2)

print(monthly_revenue.to_string())

    year  month    revenue  total_orders
0   2016      9     134.97             1
1   2016     10   40325.11           265
2   2016     12      10.90             1
3   2017      1  111798.36           750
4   2017      2  234223.40          1653
5   2017      3  359198.85          2546
6   2017      4  340669.68          2303
7   2017      5  489338.25          3546
8   2017      6  421923.37          3135
9   2017      7  481604.52          3872
10  2017      8  554699.70          4193
11  2017      9  607399.67          4150
12  2017     10  648247.65          4478
13  2017     11  987765.37          7289
14  2017     12  726033.19          5513
15  2018      1  924645.00          7069
16  2018      2  826437.13          6555
17  2018      3  953356.25          7003
18  2018      4  973534.09          6798
19  2018      5  977544.69          6749
20  2018      6  856077.86          6099
21  2018      7  867953.46          6159
22  2018      8  838576.64          6351


In [6]:
# section 3 - category revenue and freight analysis

category_data = revenue.merge(products, on="product_id")
category_data = category_data.merge(category_translation, on="product_category_name")

category_revenue = category_data.groupby("product_category_name_english").agg(
    revenue=("price", "sum"),
    freight=("freight_value", "sum"),
    total_orders=("order_id", "nunique")
).reset_index()

category_revenue["freight_percentage"] = (
    category_revenue["freight"] / category_revenue["revenue"] * 100
).round(2)

category_revenue["revenue"] = category_revenue["revenue"].round(2)
category_revenue["freight"] = category_revenue["freight"].round(2)

top10_revenue = category_revenue.sort_values("revenue", ascending=False).head(10)
top10_freight = category_revenue.sort_values("freight_percentage", ascending=False).head(10)

print("top 10 categories by revenue:")
print(top10_revenue[["product_category_name_english", "revenue", "total_orders"]].to_string())

print("\ntop 10 categories by freight percentage:")
print(top10_freight[["product_category_name_english", "freight_percentage", "revenue"]].to_string())

top 10 categories by revenue:
   product_category_name_english     revenue  total_orders
43                 health_beauty  1233131.72          8647
70                 watches_gifts  1166176.98          5495
7                 bed_bath_table  1023434.76          9272
65                sports_leisure   954852.55          7530
15         computers_accessories   888724.61          6530
39               furniture_decor   711927.69          6307
49                    housewares   615628.69          5743
20                    cool_stuff   610204.10          3559
5                           auto   578966.65          3810
69                          toys   471286.48          3804

top 10 categories by freight percentage:
        product_category_name_english  freight_percentage    revenue
46                     home_comfort_2               53.97     760.27
35                            flowers               44.04    1110.04
41  furniture_mattress_and_upholstery               36.58    4323.38
12 

In [8]:
# section 4 - delivery analysis

delivered["days_late"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days

late_orders = delivered[delivered["days_late"] > 0]
on_time_orders = delivered[delivered["days_late"] <= 0]

delivered_with_reviews = delivered.merge(
    reviews[["order_id", "review_score"]], on="order_id"
)

late_review = delivered_with_reviews[delivered_with_reviews["days_late"] > 0]["review_score"].mean()
ontime_review = delivered_with_reviews[delivered_with_reviews["days_late"] <= 0]["review_score"].mean()

delivered_with_customers = delivered.merge(
    customers[["customer_id", "customer_state"]], on="customer_id"
)

delivered_with_customers["is_late"] = delivered_with_customers["days_late"] > 0

late_by_state = delivered_with_customers.groupby("customer_state").agg(
    total_orders=("order_id", "count"),
    late_orders=("is_late", "sum")
).reset_index()

late_by_state["late_percentage"] = (
    late_by_state["late_orders"] / late_by_state["total_orders"] * 100
).round(2)

late_by_state = late_by_state.sort_values("late_percentage", ascending=False)

print("total delivered:", len(delivered))
print("on time:", len(on_time_orders))
print("late:", len(late_orders))
print("late percentage:", round(len(late_orders) / len(delivered) * 100, 2), "%")
print("avg days late:", round(late_orders["days_late"].mean(), 2))
print("orders over 7 days late:", len(late_orders[late_orders["days_late"] > 7]))
print("orders over 30 days late:", len(late_orders[late_orders["days_late"] > 30]))
print("\navg review on time:", round(ontime_review, 2))
print("avg review late:", round(late_review, 2))
print("\nlate orders by state:")
print(late_by_state.to_string())

total delivered: 96478
on time: 89936
late: 6534
late percentage: 6.77 %
avg days late: 10.62
orders over 7 days late: 2862
orders over 30 days late: 345

avg review on time: 4.29
avg review late: 2.27

late orders by state:
   customer_state  total_orders  late_orders  late_percentage
1              AL           397           85            21.41
9              MA           717          125            17.43
24             SE           335           51            15.22
16             PI           476           66            13.87
5              CE          1279          176            13.76
21             RR            41            5            12.20
4              BA          3256          396            12.16
18             RJ         12350         1495            12.11
13             PA           946          106            11.21
7              ES          1995          214            10.73
14             PB           517           54            10.44
26             TO           274

In [9]:
# section 5 - customer retention

orders_with_unique = delivered.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id"
)

customer_orders = orders_with_unique.groupby("customer_unique_id")["order_id"].count().reset_index()
customer_orders.columns = ["customer_unique_id", "total_orders"]

one_time = len(customer_orders[customer_orders["total_orders"] == 1])
repeat = len(customer_orders[customer_orders["total_orders"] > 1])
total = len(customer_orders)

print("total unique customers:", total)
print("bought only once:", one_time)
print("bought more than once:", repeat)
print("repeat customer percentage:", round(repeat / total * 100, 2), "%")
print("one time customer percentage:", round(one_time / total * 100, 2), "%")

total unique customers: 93358
bought only once: 90557
bought more than once: 2801
repeat customer percentage: 3.0 %
one time customer percentage: 97.0 %


In [11]:
# exporting data for power bi

monthly_revenue.to_csv("data/monthly_revenue.csv", index=False)
top10_revenue.to_csv("data/category_revenue.csv", index=False)
top10_freight.to_csv("data/category_freight.csv", index=False)
late_by_state.to_csv("data/late_by_state.csv", index=False)
category_revenue.to_csv("data/all_category_revenue.csv", index=False)

print("all files exported")

all files exported
